In [ ]:
!git clone https://github.com/NikhilVinod25/Automatic-Helmet-and-Number-Plate-Detection

Cloning into 'Automatic-Helmet-and-Number-Plate-Detection'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 19.99 MiB | 9.37 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
# import cv2
# import easyocr
# import numpy as np
# from ultralytics import YOLO
# import os
# import glob # This library is new - it's for finding all files in a folder
# try:
#     from google.colab.patches import cv2_imshow
# except ImportError:
#     cv2_imshow = cv2.imshow

# def main():
#     # 1. Initialize the OCR Reader (use gpu=True for Colab's GPU)
#     print("Loading EasyOCR reader...")
#     reader = easyocr.Reader(['en'], gpu=True)

#     # 2. Load Your Custom YOLOv8 Model (from Drive)
#     print("Loading custom YOLOv8 model...")
#     model_path = '/content/Automatic-Helmet-and-Number-Plate-Detection/models/best.pt'
#     model = YOLO(model_path)

#     # 3. Define Input and Output Folders
#     #
#     #    !!!! IMPORTANT: Set your input folder path !!!!
#     #    (I've set it to your validation set, which is standard)
#     #
#     input_folder = '/content/drive/MyDrive/HelmetViolations/train/images'
#     output_folder = '/content/drive/MyDrive/Project_Results1'

#     # Create the output folder if it doesn't exist
#     os.makedirs(output_folder, exist_ok=True)
#     print(f"Results will be saved in: {output_folder}")

#     # Get a list of all .jpg, .png images in the input folder
#     image_files = glob.glob(os.path.join(input_folder, '*.jpg'))
#     image_files.extend(glob.glob(os.path.join(input_folder, '*.png')))

#     if not image_files:
#         print(f"--- ERROR: No images found in {input_folder} ---")
#         return

#     print(f"Found {len(image_files)} images to process...")

#     # --- START THE LOOP ---
#     for image_path in image_files:
#         print(f"\n--- Processing: {os.path.basename(image_path)} ---")

#         img = cv2.imread(image_path)
#         if img is None:
#             print(f"Could not read image: {image_path}")
#             continue # Skip to the next image

#         # 4. Run YOLOv8 Detection
#         results = model(img)
#         result = results[0]

#         # Get the class names from the model
#         class_names = model.names
#         try:
#             names_dict = {v: k for k, v in class_names.items()}
#             plate_class_id = names_dict.get('Plate')
#         except Exception as e:
#             print(f"Error processing model classes: {e}")
#             return

#         if plate_class_id is None:
#             print("--- ERROR: 'Plate' class not found in model. ---")
#             return

#         # 5. Process Detections and Run OCR
#         detected_plates = []

#         for box in result.boxes:
#             class_id = int(box.cls)

#             if class_id == plate_class_id:
#                 coords = box.xyxy[0].cpu().numpy().astype(int)
#                 x1, y1, x2, y2 = coords

#                 cropped_plate = img[y1:y2, x1:x2]
#                 gray_plate = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2GRAY)

#                 # 6. Run OCR
#                 ocr_results = reader.readtext(gray_plate)

#                 if ocr_results:
#                     plate_text = ocr_results[0][1]
#                     detected_plates.append(plate_text)
#                     print(f"--- OCR Result: {plate_text} ---")

#         # 7. Final Report for this image
#         if not detected_plates:
#             print("--- No plates read for this image ---")

#         # 8. Save the annotated image
#         annotated_image = result.plot()

#         # Create a new unique name for the output file
#         save_name = f"result_{os.path.basename(image_path)}"
#         save_path = os.path.join(output_folder, save_name)

#         cv2.imwrite(save_path, annotated_image)

#     # --- END OF LOOP ---
#     print("\n\n--- All images processed! ---")
#     print(f"All annotated images have been saved to: {output_folder}")

# # Run the main function

# main()


In [ ]:
!pip install -r /content/Automatic-Helmet-and-Number-Plate-Detection/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 33.6 MB/s eta 0:00:00


In [ ]:
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO
import os
import glob
import re

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = cv2.imshow

def clean_plate_text(text):
    """
    Clean and extract only alphanumeric characters from OCR result
    """
    # Remove special characters and keep only letters and numbers
    cleaned = re.sub(r'[^A-Z0-9]', '', text.upper())
    return cleaned

def process_frame(img, model, reader, plate_class_id):
    """
    Process a single frame (works for both image and video)
    Returns: annotated image and list of detected plate texts
    """
    # Run YOLOv8 Detection
    results = model(img)
    result = results[0]

    detected_plates = []

    # Process each detection
    for box in result.boxes:
        class_id = int(box.cls)

        if class_id == plate_class_id:
            coords = box.xyxy[0].cpu().numpy().astype(int)
            x1, y1, x2, y2 = coords

            # Crop the plate region
            cropped_plate = img[y1:y2, x1:x2]

            # Preprocess for better OCR
            gray_plate = cv2.cvtColor(cropped_plate, cv2.COLOR_BGR2GRAY)

            # Apply preprocessing to improve OCR accuracy
            # Increase contrast
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            enhanced = clahe.apply(gray_plate)

            # Denoise
            denoised = cv2.fastNlMeansDenoising(enhanced)

            # Threshold
            _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # Run OCR on preprocessed image
            ocr_results = reader.readtext(thresh, detail=0)

            if ocr_results:
                # Combine all detected text and clean it
                raw_text = ' '.join(ocr_results)
                plate_text = clean_plate_text(raw_text)

                if plate_text:  # Only add if we got valid text
                    detected_plates.append(plate_text)

                    # Draw the plate text on the image
                    cv2.putText(img, plate_text, (x1, y1 - 10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Get annotated image with bounding boxes
    annotated_image = result.plot()

    # Add plate texts to the annotated image
    for i, plate_text in enumerate(detected_plates):
        cv2.putText(annotated_image, plate_text, (10, 30 + i*30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    return annotated_image, detected_plates

def process_image(image_path, model, reader, plate_class_id, output_folder):
    """
    Process a single image
    """
    print(f"\n--- Processing Image: {os.path.basename(image_path)} ---")

    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not read image: {image_path}")
        return

    annotated_image, detected_plates = process_frame(img, model, reader, plate_class_id)

    # Print results
    if detected_plates:
        print(f"Detected Plates: {', '.join(detected_plates)}")
    else:
        print("No plates detected")

    # Save the annotated image
    save_name = f"result_{os.path.basename(image_path)}"
    save_path = os.path.join(output_folder, save_name)
    cv2.imwrite(save_path, annotated_image)
    print(f"Saved to: {save_path}")

def process_video(video_path, model, reader, plate_class_id, output_folder):
    """
    Process a video file
    """
    print(f"\n--- Processing Video: {os.path.basename(video_path)} ---")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Define output video path
    save_name = f"result_{os.path.basename(video_path)}"
    save_path = os.path.join(output_folder, save_name)

    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, fps, (width, height))

    frame_count = 0
    all_detected_plates = set()  # Use set to avoid duplicates

    print(f"Total frames: {total_frames}, FPS: {fps}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # Process frame
        annotated_frame, detected_plates = process_frame(frame, model, reader, plate_class_id)

        # Collect unique plates
        all_detected_plates.update(detected_plates)

        # Write frame to output video
        out.write(annotated_frame)

        # Print progress every 30 frames
        if frame_count % 30 == 0:
            print(f"Processed {frame_count}/{total_frames} frames...", end='\r')

    # Release resources
    cap.release()
    out.release()

    print(f"\nVideo processing complete!")
    print(f"Unique plates detected in video: {', '.join(all_detected_plates) if all_detected_plates else 'None'}")
    print(f"Saved to: {save_path}")

def main():
    # 1. Initialize the OCR Reader
    print("Loading EasyOCR reader...")
    reader = easyocr.Reader(['en'], gpu=True)

    # 2. Load Your Custom YOLOv8 Model
    print("Loading custom YOLOv8 model...")
    model_path = '/content/Automatic-Helmet-and-Number-Plate-Detection/models/best.pt'
    model = YOLO(model_path)

    # 3. Define Input and Output Folders
    input_folder = '/content/Automatic-Helmet-and-Number-Plate-Detection/data'
    output_folder = '/content/Automatic-Helmet-and-Number-Plate-Detection/data'

    # Create the output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    print(f"Results will be saved in: {output_folder}")

    # Get class ID for 'Plate'
    class_names = model.names
    try:
        names_dict = {v: k for k, v in class_names.items()}
        plate_class_id = names_dict.get('Plate')
    except Exception as e:
        print(f"Error processing model classes: {e}")
        return

    if plate_class_id is None:
        print("--- ERROR: 'Plate' class not found in model. ---")
        return

    # 4. Get all image and video files
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    video_extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv']

    image_files = []
    video_files = []

    # Collect image files
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(input_folder, ext)))

    # Collect video files
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(input_folder, ext)))

    total_files = len(image_files) + len(video_files)

    if total_files == 0:
        print(f"--- ERROR: No images or videos found in {input_folder} ---")
        return

    print(f"\nFound {len(image_files)} images and {len(video_files)} videos to process...")

    # 5. Process all images
    for image_path in image_files:
        process_image(image_path, model, reader, plate_class_id, output_folder)

    # 6. Process all videos
    for video_path in video_files:
        process_video(video_path, model, reader, plate_class_id, output_folder)

    print("\n\n=== All files processed! ===")
    print(f"All results saved to: {output_folder}")

# Run the main function
if __name__ == "__main__":
    main()

Loading EasyOCR reader...
Loading custom YOLOv8 model...
Results will be saved in: /content/Automatic-Helmet-and-Number-Plate-Detection/data

Found 2 images and 0 videos to process...

--- Processing Image: no_helmet.jpg ---

0: 640x416 1 WithoutHelmet, 12.3ms
Speed: 3.2ms preprocess, 12.3ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 416)
No plates detected
Saved to: /content/Automatic-Helmet-and-Number-Plate-Detection/data/result_no_helmet.jpg

--- Processing Image: test.png ---

0: 384x640 2 Plates, 4 WithoutHelmets, 11.4ms
Speed: 1.8ms preprocess, 11.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Detected Plates: 1793, EURD
Saved to: /content/Automatic-Helmet-and-Number-Plate-Detection/data/result_test.png


=== All files processed! ===
All results saved to: /content/Automatic-Helmet-and-Number-Plate-Detection/data


# BEST SOLUTION

In [ ]:
import cv2
import easyocr
import numpy as np
from ultralytics import YOLO
import os
import glob
import re

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    cv2_imshow = cv2.imshow

def clean_plate_text(text):
    """
    Clean and extract only alphanumeric characters from OCR result
    """
    # Remove special characters and keep only letters and numbers
    cleaned = re.sub(r'[^A-Z0-9]', '', text.upper())
    return cleaned

def process_frame(img, model, reader, plate_class_id, crops_folder, frame_identifier):
    """
    Process a single frame (works for both image and video)
    Returns: annotated image and list of detected plate texts
    """
    # Run YOLOv8 Detection with best.pt
    results = model(img)
    result = results[0]

    detected_plates = []
    plate_counter = 0

    print(f"  🔍 Detected {len(result.boxes)} objects")

    # Process each detection
    for box in result.boxes:
        class_id = int(box.cls)
        confidence = float(box.conf)
        class_name = model.names[class_id]

        if class_id == plate_class_id:
            print(f"  🎯 Found plate with confidence: {confidence:.2f}")

            coords = box.xyxy[0].cpu().numpy().astype(int)
            x1, y1, x2, y2 = coords

            # Add padding to crop
            padding = 15
            x1_padded = max(0, x1 - padding)
            y1_padded = max(0, y1 - padding)
            x2_padded = min(img.shape[1], x2 + padding)
            y2_padded = min(img.shape[0], y2 + padding)

            # Crop the plate region
            cropped_plate = img[y1_padded:y2_padded, x1_padded:x2_padded]

            # Skip if crop is too small
            if cropped_plate.shape[0] < 20 or cropped_plate.shape[1] < 20:
                print(f"    ⚠️ Plate crop too small ({cropped_plate.shape[1]}x{cropped_plate.shape[0]}), skipping")
                continue

            plate_counter += 1

            print(f"    📏 Plate crop size: {cropped_plate.shape[1]}x{cropped_plate.shape[0]}")

            # Save the cropped plate
            crop_filename = f"{frame_identifier}_plate_{plate_counter}.jpg"
            crop_path = os.path.join(crops_folder, crop_filename)
            cv2.imwrite(crop_path, cropped_plate)

            # Resize if too small for better OCR
            if cropped_plate.shape[1] < 200:
                scale = 200 / cropped_plate.shape[1]
                new_width = int(cropped_plate.shape[1] * scale)
                new_height = int(cropped_plate.shape[0] * scale)
                cropped_plate = cv2.resize(cropped_plate, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
                print(f"    🔍 Upscaled to: {new_width}x{new_height}")

            # Run OCR on ORIGINAL image (no preprocessing)
            try:
                print(f"    🔬 Running OCR on original image...")

                # OCR on the original cropped plate
                ocr_results = reader.readtext(cropped_plate, detail=0)

                plate_text = None

                if ocr_results:
                    # Combine all detected text
                    raw_text = ' '.join(ocr_results)
                    plate_text = clean_plate_text(raw_text)

                    print(f"    📝 OCR raw text: {raw_text}")
                    print(f"    ✅ Cleaned text: {plate_text}")

                if plate_text and len(plate_text) >= 4:
                    detected_plates.append(plate_text)

                    # Save crop with plate text in filename
                    crop_filename_with_text = f"{frame_identifier}_plate_{plate_counter}_{plate_text}.jpg"
                    crop_path_with_text = os.path.join(crops_folder, crop_filename_with_text)

                    # Add text overlay to crop
                    crop_with_text = cropped_plate.copy()

                    # Calculate text size and position
                    font_scale = min(cropped_plate.shape[1] / 150, 1.2)
                    thickness = max(1, int(font_scale * 2))

                    cv2.putText(crop_with_text, plate_text, (10, 30),
                               cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), thickness)
                    cv2.imwrite(crop_path_with_text, crop_with_text)

                    print(f"    🎯 ✅ PLATE NUMBER: {plate_text}")
                else:
                    print(f"    ❌ No valid plate text found (got: {ocr_results})")

            except Exception as e:
                print(f"    ❌ OCR Error: {str(e)}")
                import traceback
                traceback.print_exc()

    # Get annotated image with bounding boxes
    annotated_image = result.plot()

    # Add plate text labels on the annotated image
    for i, (box, text) in enumerate(zip([b for b in result.boxes if int(b.cls) == plate_class_id], detected_plates)):
        coords = box.xyxy[0].cpu().numpy().astype(int)
        x1, y1, x2, y2 = coords

        # Draw plate number above the box
        label_y = max(y1 - 10, 20)
        cv2.putText(annotated_image, text, (x1, label_y),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)

    return annotated_image, detected_plates

def process_image(image_path, model, reader, plate_class_id, output_folder, crops_folder):
    """
    Process a single image
    """
    print(f"\n{'='*70}")
    print(f"📸 Processing Image: {os.path.basename(image_path)}")
    print(f"{'='*70}")

    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Could not read image: {image_path}")
        return

    print(f"  📐 Image size: {img.shape[1]}x{img.shape[0]}")

    frame_identifier = os.path.splitext(os.path.basename(image_path))[0]

    annotated_image, detected_plates = process_frame(img, model, reader,
                                                      plate_class_id, crops_folder, frame_identifier)

    # Print results
    print(f"\n{'='*70}")
    if detected_plates:
        print(f"🚗 DETECTED NUMBER PLATES:")
        for i, plate in enumerate(detected_plates, 1):
            print(f"   {i}. {plate}")
    else:
        print("❌ No number plates with readable text detected")
    print(f"{'='*70}")

    # Save the annotated image
    save_name = f"result_{os.path.basename(image_path)}"
    save_path = os.path.join(output_folder, save_name)
    cv2.imwrite(save_path, annotated_image)
    print(f"💾 Saved: {save_name}\n")

def process_video(video_path, model, reader, plate_class_id, output_folder, crops_folder):
    """
    Process a video file
    """
    print(f"\n{'='*70}")
    print(f"🎬 Processing Video: {os.path.basename(video_path)}")
    print(f"{'='*70}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Could not open video: {video_path}")
        return

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Define output video path
    save_name = f"result_{os.path.basename(video_path)}"
    save_path = os.path.join(output_folder, save_name)

    # Define codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(save_path, fourcc, fps, (width, height))

    frame_count = 0
    all_detected_plates = set()  # Use set to avoid duplicates

    video_identifier = os.path.splitext(os.path.basename(video_path))[0]

    print(f"📹 Total frames: {total_frames}, FPS: {fps}")
    print(f"🔍 Processing every 15th frame for OCR (to save time)...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # Process every 15th frame with OCR (to save time)
        if frame_count % 15 == 0:
            print(f"\n--- Frame {frame_count}/{total_frames} ({int(frame_count/total_frames*100)}%) ---")
            frame_identifier = f"{video_identifier}_frame{frame_count}"
            annotated_frame, detected_plates = process_frame(frame, model, reader,
                                                             plate_class_id, crops_folder, frame_identifier)
            # Collect unique plates
            if detected_plates:
                all_detected_plates.update(detected_plates)
                print(f"✅ Plates in this frame: {', '.join(detected_plates)}")
        else:
            # Just run detection without OCR for other frames
            results = model(frame, verbose=False)
            annotated_frame = results[0].plot()

        # Write frame to output video
        out.write(annotated_frame)

    # Release resources
    cap.release()
    out.release()

    print(f"\n{'='*70}")
    print(f"✅ Video processing complete!")
    if all_detected_plates:
        print(f"🚗 ALL UNIQUE NUMBER PLATES DETECTED IN VIDEO:")
        for i, plate in enumerate(sorted(all_detected_plates), 1):
            print(f"   {i}. {plate}")
    else:
        print("❌ No number plates with readable text detected in video")
    print(f"💾 Saved: {save_name}")
    print(f"{'='*70}\n")

def main():
    print("\n" + "="*70)
    print("🚀 HELMET & NUMBER PLATE DETECTION SYSTEM")
    print("   - Detection Model: best.pt (Helmet + Plate)")
    print("   - OCR Method: Original (no preprocessing)")
    print("="*70 + "\n")

    # 1. Initialize the OCR Reader
    print("📦 Loading EasyOCR reader...")
    reader = easyocr.Reader(['en'], gpu=True)
    print("✅ EasyOCR loaded\n")

    # 2. Load Your Custom YOLOv8 Model (best.pt)
    print("📦 Loading custom YOLOv8 model (best.pt)...")
    model_path = '/content/Automatic-Helmet-and-Number-Plate-Detection/models/best.pt'
    model = YOLO(model_path)
    print(f"✅ Model loaded: {model_path}")
    print(f"   Classes: {model.names}\n")

    # 3. Define Input and Output Folders
    input_folder = '/content/Automatic-Helmet-and-Number-Plate-Detection/data'
    output_folder = '/content/Automatic-Helmet-and-Number-Plate-Detection/results'
    crops_folder = '/content/Automatic-Helmet-and-Number-Plate-Detection/plate_crops'

    # Create folders
    os.makedirs(input_folder, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(crops_folder, exist_ok=True)

    print(f"📂 Input: {input_folder}")
    print(f"📂 Output: {output_folder}")
    print(f"📂 Plate crops: {crops_folder}\n")

    # Get class ID for 'Plate'
    class_names = model.names
    try:
        names_dict = {v: k for k, v in class_names.items()}
        plate_class_id = names_dict.get('Plate')
    except Exception as e:
        print(f"❌ Error processing model classes: {e}")
        return

    if plate_class_id is None:
        print(f"❌ ERROR: 'Plate' class not found in model")
        print(f"Available classes: {list(class_names.values())}")
        return

    print(f"✅ Plate class ID: {plate_class_id}\n")

    # 4. Get all image and video files
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    video_extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv']

    image_files = []
    video_files = []

    # Collect image files
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(input_folder, ext)))

    # Collect video files
    for ext in video_extensions:
        video_files.extend(glob.glob(os.path.join(input_folder, ext)))

    total_files = len(image_files) + len(video_files)

    if total_files == 0:
        print(f"❌ No images or videos found in {input_folder}")
        print("Please upload your files to this folder")
        return

    print(f"📊 Found: {len(image_files)} images and {len(video_files)} videos\n")

    # 5. Process all images
    if len(image_files) > 0:
        print(f"{'='*70}")
        print(f"🖼️  PROCESSING {len(image_files)} IMAGES")
        print(f"{'='*70}")

        for i, image_path in enumerate(image_files, 1):
            print(f"\n[IMAGE {i}/{len(image_files)}]")
            process_image(image_path, model, reader, plate_class_id, output_folder, crops_folder)

    # 6. Process all videos
    if len(video_files) > 0:
        print(f"\n{'='*70}")
        print(f"🎬 PROCESSING {len(video_files)} VIDEOS")
        print(f"{'='*70}")

        for i, video_path in enumerate(video_files, 1):
            print(f"\n[VIDEO {i}/{len(video_files)}]")
            process_video(video_path, model, reader, plate_class_id, output_folder, crops_folder)

    print("\n" + "="*70)
    print("✅ ALL FILES PROCESSED!")
    print(f"📁 Results saved to: {output_folder}")
    print(f"📁 Plate crops saved to: {crops_folder}")
    print("="*70 + "\n")

# Run the main function
if __name__ == "__main__":
    main()


🚀 HELMET & NUMBER PLATE DETECTION SYSTEM
   - Detection Model: best.pt (Helmet + Plate)
   - OCR Method: Original (no preprocessing)

📦 Loading EasyOCR reader...
✅ EasyOCR loaded

📦 Loading custom YOLOv8 model (best.pt)...
✅ Model loaded: /content/Automatic-Helmet-and-Number-Plate-Detection/models/best.pt
   Classes: {0: 'Plate', 1: 'WithHelmet', 2: 'WithoutHelmet'}

📂 Input: /content/Automatic-Helmet-and-Number-Plate-Detection/data
📂 Output: /content/Automatic-Helmet-and-Number-Plate-Detection/results
📂 Plate crops: /content/Automatic-Helmet-and-Number-Plate-Detection/plate_crops

✅ Plate class ID: 0

📊 Found: 2 images and 0 videos

🖼️  PROCESSING 2 IMAGES

[IMAGE 1/2]

📸 Processing Image: no_helmet.jpg
  📐 Image size: 870x1390

0: 640x416 1 WithoutHelmet, 13.0ms
Speed: 2.5ms preprocess, 13.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 416)
  🔍 Detected 1 objects

❌ No number plates with readable text detected
💾 Saved: result_no_helmet.jpg


[IMAGE 2/2]

📸 Processin